# Find the Moment

This notebook runs the repository; it does not contain a second copy of the application. In Colab, select a T4 GPU. Upload the repository ZIP when prompted, or set REPO_URL after publishing on GitHub. In VS Code, open the extracted repository and use its Python environment.


## Load the repository

In [ ]:
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

# After publishing on GitHub, set this URL. Leave it empty to upload the ZIP.
REPO_URL = ""

try:
    from google.colab import files

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    destination = Path("/content/find-the-moment-repo")
    if REPO_URL:
        if not destination.exists():
            subprocess.run(["git", "clone", REPO_URL, str(destination)], check=True)
        project = destination
    else:
        uploaded = files.upload()
        archive_path = next((Path(name) for name in uploaded if name.endswith(".zip")), None)
        if archive_path is None:
            raise ValueError("Upload the find-the-moment repository ZIP.")
        destination.mkdir(exist_ok=True)
        with zipfile.ZipFile(archive_path) as archive:
            for item in archive.infolist():
                target = (destination / item.filename).resolve()
                if not target.is_relative_to(destination.resolve()):
                    raise ValueError("Invalid archive path.")
            archive.extractall(destination)
        project = next(
            (
                p.parent
                for p in destination.rglob("pyproject.toml")
                if (p.parent / "find_the_moment").is_dir()
            ),
            None,
        )
        if project is None:
            raise ValueError("The ZIP does not contain the application package.")
else:
    project = next(
        (
            p
            for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").is_file() and (p / "find_the_moment").is_dir()
        ),
        None,
    )
    if project is None:
        raise ValueError("Open this notebook from the extracted repository folder.")

print("Project:", project)

## Install dependencies

In [ ]:
import sys

if IN_COLAB:
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "torch==2.8.0",
            "torchvision==0.23.0",
            "torchaudio==2.8.0",
            "--index-url",
            "https://download.pytorch.org/whl/cu126",
        ],
        check=True,
    )
elif shutil.which("ffmpeg") is None:
    raise RuntimeError("Install FFmpeg first. On macOS: brew install ffmpeg")

subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(project)], check=True)
print("Dependencies installed. The launch cell uses a fresh Python process.")

## Start the app

Open the printed URL. Colab prints login credentials for the share link. Keep this cell running and stop it before relaunching. Model downloads begin on first use.


In [ ]:
# Stop any child process started by an earlier run of this cell.
previous_server = globals().get("server")
if previous_server is not None and previous_server.poll() is None:
    previous_server.terminate()
    try:
        previous_server.wait(timeout=10)
    except subprocess.TimeoutExpired:
        previous_server.kill()
        previous_server.wait()

environment = dict(os.environ)
environment["MAX_DURATION_SECONDS"] = "7200"
if IN_COLAB:
    environment["COLAB_SHARE"] = "1"
    environment["MEDIA_LIBRARY_DIR"] = "/content/recording_library"
else:
    environment.pop("COLAB_SHARE", None)
    environment["MEDIA_LIBRARY_DIR"] = str(project / "recording_library")

server = subprocess.Popen(
    [sys.executable, "-u", str(project / "app.py")],
    cwd=project,
    env=environment,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
try:
    for line in server.stdout:
        print(line, end="", flush=True)
    if server.wait() != 0:
        raise RuntimeError("The app exited with an error. Check the output above.")
except KeyboardInterrupt:
    server.terminate()
    try:
        server.wait(timeout=10)
    except subprocess.TimeoutExpired:
        server.kill()
        server.wait()
    print("App stopped.")